In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv("10-diamonds.csv")

In [3]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
df.drop("Unnamed: 0", axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


In [5]:
df.drop(df[df["x"]==0].index, inplace=True)
df.drop(df[df["y"]==0].index, inplace=True)
df.drop(df[df["z"]==0].index, inplace=True)

In [6]:
df = df[((df["x"]<10))]
df = df[((df["y"]<20))]
df = df[((df["z"]<10) & (df["z"]>2))]
df = df[((df["depth"]<75) & (df["depth"]>45))]
df = df[((df["table"]<72) & (df["table"]>48))]

In [7]:
X = df.drop("price",axis = 1)
y =df["price"]

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=15)

In [10]:
from sklearn.preprocessing import LabelEncoder

In [12]:
#her kolon için ayrı bir enoder oluşturacağız çünkü encoder ve scalerları model ile birlikte kaydetmemiz gerekecek.
encoders = {}

for col in ["cut","color","clarity"]:
    encoders[col] = LabelEncoder()
    X_train[col]= encoders[col].fit_transform(X_train[col])
    X_test[col]= encoders[col].transform(X_test[col])

In [14]:
X_train.head()

,carat,cut,color,clarity,depth,table,x,y,z
4850,0.71,2,2,4,61.4,56.0,5.75,5.78,3.54
33334,0.32,2,2,6,61.1,56.0,4.42,4.45,2.71
37740,0.41,2,1,4,60.5,58.0,4.81,4.85,2.92
39730,0.45,3,1,2,62.8,58.0,4.88,4.84,3.05
5506,0.91,3,1,3,61.6,60.0,6.14,6.10,3.77


In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.svm import SVR

In [17]:
svr = SVR(C=1000, gamma=0.1, kernel='rbf')
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

In [18]:
from sklearn.metrics import r2_score
score = r2_score(y_test, y_pred)
print("R2 score:",score)

R2 score: 0.943032078815135


In [19]:
encoders

{'cut': LabelEncoder(), 'color': LabelEncoder(), 'clarity': LabelEncoder()}

In [20]:
scaler

,copy,True
,with_mean,True
,with_std,True


In [21]:
svr

,kernel,'rbf'
,degree,3
,gamma,0.1
,coef0,0.0
,tol,0.001
,C,1000
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [22]:
import pickle

In [23]:
with open("diamond_model.pkl", "wb") as f:
    pickle.dump(
        {
            "scaler": scaler,
            "model": svr,
            "encoders": encoders
        }
    ,f)

In [24]:
pd.DataFrame(X_train_scaled).to_csv("Diamonds.csv",index=False)